In [1]:
# ================================================================
# WEEK 7 — DAY 4
# FINAL RAG DOCUMENT QA SYSTEM
# Real PDF Upload • Hybrid Retrieval • Gradio Interface • ngrok Deployment
# ================================================================
#
# WEEK 7 PROJECT:
# RAG-Based Document Q&A System
#
# DAY 4 GOAL:
# Build a production-ready RAG system with a web interface.
#
# By the end of this notebook, we will build:
#
#     User Uploads PDF
#          ↓
#     PyMuPDF Extracts Text
#          ↓
#     Document Chunking
#          ↓
#     ┌─────────────┼─────────────┐
#     ↓             ↓             ↓
#  FAISS Dense   BM25 Sparse    Hybrid RRF
#     ↓             ↓             ↓
#     └─────────────┼─────────────┘
#                   ↓
#              Reranking (MMR)
#                   ↓
#              LLM Generation
#                   ↓
#              Citations + Answer
#                   ↓
#              Gradio Web UI
#
# CONCEPTS WE WILL LEARN:
#
# 1. PDF text extraction with PyMuPDF
# 2. Building RAG pipeline for custom documents
# 3. Hybrid retrieval with FAISS + BM25
# 4. MMR reranking for diverse results
# 5. LLM-based answer generation
# 6. Gradio web interface creation
# 7. End-to-end RAG system
#
# DEPENDENCY POLICY:
#
# This notebook does NOT depend on:
#     - Day 1, 2, or 3 notebooks
#     - Previous embeddings or indexes
#     - Previous Kaggle sessions
#
# Run this notebook from top to bottom in a fresh Kaggle
# environment and it should work independently.
#
# ================================================================

print("=" * 70)
print("WEEK 7 — DAY 4: FINAL RAG DOCUMENT QA SYSTEM")
print("=" * 70)
print()
print("Focus: Real PDF Upload + Hybrid RAG + Gradio")
print("Goal: Production-ready RAG web application")
print()
print("Notebook Status: Standalone")
print("=" * 70)

WEEK 7 — DAY 4: FINAL RAG DOCUMENT QA SYSTEM

Focus: Real PDF Upload + Hybrid RAG + Gradio
Goal: Production-ready RAG web application

Notebook Status: Standalone


In [2]:
# Install required packages for Day 4

!pip install -q \
    gradio \
    pymupdf \
    sentence-transformers \
    faiss-cpu \
    rank-bm25 \
    transformers \
    torch

print("All dependencies installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 72.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.4 MB/s eta 0:00:00:00:0100:01
All dependencies installed successfully.


In [3]:
# Standard library imports
import os
import re
import random
import time
import tempfile
from typing import List, Dict, Tuple, Optional, Any
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Data manipulation
import numpy as np

# Machine learning
import torch

# PDF processing
import pymupdf

# Embeddings
from sentence_transformers import SentenceTransformer

# Vector search
import faiss

# BM25
from rank_bm25 import BM25Okapi

# LLM
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Gradio
import gradio as gr

print("All imports completed successfully.")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

All imports completed successfully.
PyTorch version: 2.10.0+cu128
Device: cuda


In [4]:
# Set reproducibility and configuration values

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Model configuration - optimized for Kaggle
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_NAME = "google/flan-t5-small"

# Text chunking configuration
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# Retrieval configuration
INITIAL_RETRIEVAL_K = 20
FINAL_K = 5
RRF_CONSTANT = 60

print("\nConfiguration loaded successfully.")
print(f"Device: {DEVICE}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"LLM model: {LLM_MODEL_NAME}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Chunk overlap: {CHUNK_OVERLAP}")
print(f"Initial K: {INITIAL_RETRIEVAL_K}")
print(f"Final K: {FINAL_K}")
print("=" * 50)

Using device: cuda

Configuration loaded successfully.
Device: cuda
Embedding model: sentence-transformers/all-MiniLM-L6-v2
LLM model: google/flan-t5-small
Chunk size: 500
Chunk overlap: 50
Initial K: 20
Final K: 5


In [15]:
class RAGSystem:
    """
    Complete RAG system with hybrid retrieval and LLM generation.
    """
    
    def __init__(self):
        self.device = DEVICE
        self.chunk_size = CHUNK_SIZE
        self.chunk_overlap = CHUNK_OVERLAP
        self.initial_k = INITIAL_RETRIEVAL_K
        self.final_k = FINAL_K
        self.rrf_constant = RRF_CONSTANT
        
        self.chunks = []
        self.chunk_texts = []
        self.embeddings = None
        self.faiss_index = None
        self.bm25_index = None
        self.tokenized_chunks = []
        
        self.embedding_model = None
        self.tokenizer = None
        self.model = None
        self.llm_available = False
        
        # Load models
        self._load_models()
    
    def _load_models(self):
        """Load embedding and LLM models."""
        print("Loading embedding model...")
        self.embedding_model = SentenceTransformer(
            EMBEDDING_MODEL_NAME,
            device=self.device
        )
        print(f"Embedding model loaded. Dimension: {self.embedding_model.get_sentence_embedding_dimension()}")
        
        print("Loading LLM...")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL_NAME)
            
            if self.device == "cuda":
                self.model = self.model.to(self.device)
            
            self.llm_available = True
            print(f"LLM loaded successfully: {LLM_MODEL_NAME}")
            
        except Exception as e:
            print(f"LLM load failed: {e}")
            print("Using retrieval-only mode")
            self.llm_available = False
    
    def _clean_text(self, text: str) -> str:
        """Clean extracted text."""
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    
    def _create_chunks(self, text: str) -> List[str]:
        """Split text into overlapping chunks with smaller size for better accuracy."""
        # Use smaller chunk size for better retrieval accuracy
        chunk_size = 300  # Reduced from 500
        overlap = 50
        
        chunks = []
        start = 0
        text_length = len(text)
        
        while start < text_length:
            end = min(start + chunk_size, text_length)
            chunk = text[start:end]
            chunks.append(chunk)
            start += (chunk_size - overlap)
            if end == text_length:
                break
        
        return chunks
    
    def _tokenize_bm25(self, text: str) -> List[str]:
        """Tokenize text for BM25."""
        return re.findall(r'\w+', text.lower())
    
    def _build_faiss_index(self, embeddings: np.ndarray):
        """Build FAISS index for dense retrieval."""
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatIP(dimension)
        faiss.normalize_L2(embeddings)
        index.add(embeddings.astype('float32'))
        return index
    
    def _build_bm25_index(self, tokenized_chunks: List[List[str]]):
        """Build BM25 index for sparse retrieval."""
        return BM25Okapi(tokenized_chunks)
    
    def _reciprocal_rank_fusion(self, dense_scores: List[float], sparse_scores: List[float]) -> List[int]:
        """Combine dense and sparse retrieval with RRF."""
        dense_ranking = {idx: rank+1 for rank, idx in enumerate(np.argsort(dense_scores)[::-1])}
        sparse_ranking = {idx: rank+1 for rank, idx in enumerate(np.argsort(sparse_scores)[::-1])}
        
        all_indices = set(dense_ranking.keys()) | set(sparse_ranking.keys())
        rrf_scores = {}
        
        for idx in all_indices:
            score = 0
            if idx in dense_ranking:
                score += 1 / (self.rrf_constant + dense_ranking[idx])
            if idx in sparse_ranking:
                score += 1 / (self.rrf_constant + sparse_ranking[idx])
            rrf_scores[idx] = score
        
        return sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    
    def _mmr_rerank(self, query: str, candidate_indices: List[int], k: int) -> List[int]:
        """Rerank results using MMR (Maximum Marginal Relevance)."""
        valid_indices = [idx for idx in candidate_indices if idx < len(self.chunk_texts)]
        
        if not valid_indices:
            return []
        
        if len(valid_indices) <= k:
            return valid_indices[:k]
        
        candidate_texts = [self.chunk_texts[idx] for idx in valid_indices]
        candidate_embeddings = self.embedding_model.encode(
            candidate_texts, device=self.device, show_progress_bar=False
        )
        
        query_embedding = self.embedding_model.encode(
            [query], device=self.device, show_progress_bar=False
        )
        
        faiss.normalize_L2(candidate_embeddings)
        faiss.normalize_L2(query_embedding)
        
        relevance_scores = np.dot(candidate_embeddings, query_embedding.T).flatten()
        
        selected_indices = []
        remaining_indices = list(range(len(valid_indices)))
        lambda_param = 0.5  # Balanced relevance and diversity
        
        for _ in range(min(k, len(valid_indices))):
            mmr_scores = []
            for idx in remaining_indices:
                relevance = relevance_scores[idx]
                if selected_indices:
                    similarity_to_selected = np.max(np.dot(
                        candidate_embeddings[idx:idx+1],
                        candidate_embeddings[selected_indices].T
                    ))
                else:
                    similarity_to_selected = 0
                mmr_score = lambda_param * relevance - (1 - lambda_param) * similarity_to_selected
                mmr_scores.append((idx, mmr_score))
            
            best_idx = max(mmr_scores, key=lambda x: x[1])[0]
            selected_indices.append(best_idx)
            remaining_indices.remove(best_idx)
        
        return [valid_indices[idx] for idx in selected_indices]
    
    def _expand_context(self, query: str, chunks: List[str], max_chunks: int = 5) -> str:
        """Expand context by finding additional relevant sentences."""
        # Get query terms
        query_terms = set(query.lower().split())
        
        # Score each chunk
        chunk_scores = []
        for i, chunk in enumerate(chunks):
            chunk_lower = chunk.lower()
            # Count query term matches
            term_matches = sum(1 for term in query_terms if term in chunk_lower)
            # Bonus for exact phrase
            phrase_bonus = 2 if query.lower() in chunk_lower else 0
            score = term_matches + phrase_bonus
            chunk_scores.append((i, score, chunk))
        
        # Sort by score
        chunk_scores.sort(key=lambda x: x[1], reverse=True)
        
        # Take top chunks
        top_chunks = [chunk for _, score, chunk in chunk_scores[:max_chunks] if score > 0]
        
        if not top_chunks:
            return "\n\n".join(chunks[:max_chunks])
        
        return "\n\n".join(top_chunks)
    
    def process_pdf(self, pdf_path: str) -> Dict:
        """Process a PDF file and build the RAG index."""
        print(f"\nProcessing PDF: {pdf_path}")
        start_time = time.time()
        
        try:
            doc = pymupdf.open(pdf_path)
            all_text = ""
            for page in doc:
                text = page.get_text()
                if text:
                    all_text += text + "\n"
            doc.close()
        except Exception as e:
            return {"success": False, "error": f"PDF extraction failed: {e}"}
        
        if not all_text.strip():
            return {"success": False, "error": "No text extracted from PDF"}
        
        all_text = self._clean_text(all_text)
        chunks = self._create_chunks(all_text)
        
        if not chunks:
            return {"success": False, "error": "No chunks created"}
        
        self.chunks = chunks
        self.chunk_texts = chunks
        
        print(f"Created {len(chunks)} chunks")
        print(f"Total text length: {len(all_text)} characters")
        
        print("Generating embeddings...")
        embeddings = self.embedding_model.encode(
            chunks, batch_size=32, device=self.device, show_progress_bar=True
        )
        self.embeddings = embeddings
        
        print("Building FAISS index...")
        self.faiss_index = self._build_faiss_index(embeddings)
        
        print("Building BM25 index...")
        tokenized = [self._tokenize_bm25(chunk) for chunk in chunks]
        self.tokenized_chunks = tokenized
        self.bm25_index = self._build_bm25_index(tokenized)
        
        elapsed_time = time.time() - start_time
        
        return {
            "success": True,
            "num_chunks": len(chunks),
            "embedding_dimension": embeddings.shape[1],
            "elapsed_time": elapsed_time,
            "text_length": len(all_text)
        }
    
    def query(self, question: str) -> Dict:
        """Answer a question using the RAG system."""
        if self.faiss_index is None or self.bm25_index is None:
            return {"error": "No document loaded. Please upload a PDF first."}
        
        if not question or not question.strip():
            return {"error": "Please enter a question."}
        
        start_time = time.time()
        
        # Dense retrieval - get more candidates
        query_embedding = self.embedding_model.encode(
            [question], device=self.device, show_progress_bar=False
        )
        faiss.normalize_L2(query_embedding)
        dense_scores, dense_indices = self.faiss_index.search(
            query_embedding.astype('float32'), min(self.initial_k, len(self.chunk_texts))
        )
        dense_scores = dense_scores[0].tolist()
        dense_indices = dense_indices[0].tolist()
        
        # Sparse retrieval
        query_tokens = self._tokenize_bm25(question)
        bm25_scores = self.bm25_index.get_scores(query_tokens)
        
        # Get top indices from BM25
        k = min(self.initial_k, len(self.chunk_texts))
        sparse_indices = np.argsort(bm25_scores)[-k:][::-1].tolist()
        sparse_scores = [bm25_scores[idx] for idx in sparse_indices]
        
        # Ensure indices are valid
        valid_dense_indices = [idx for idx in dense_indices if idx < len(self.chunk_texts)]
        valid_sparse_indices = [idx for idx in sparse_indices if idx < len(self.chunk_texts)]
        
        if not valid_dense_indices and not valid_sparse_indices:
            return {"error": "No valid chunks found for retrieval."}
        
        # Hybrid RRF
        if valid_dense_indices and valid_sparse_indices:
            rrf_indices = self._reciprocal_rank_fusion(dense_scores, sparse_scores)
        elif valid_dense_indices:
            rrf_indices = valid_dense_indices
        else:
            rrf_indices = valid_sparse_indices
        
        hybrid_indices = rrf_indices[:self.initial_k]
        hybrid_indices = [idx for idx in hybrid_indices if idx < len(self.chunk_texts)]
        
        if not hybrid_indices:
            return {"error": "No valid indices after hybrid retrieval."}
        
        # MMR reranking - get more candidates for better selection
        mmr_indices = self._mmr_rerank(question, hybrid_indices, min(self.final_k * 2, len(hybrid_indices)))
        
        if not mmr_indices:
            mmr_indices = hybrid_indices[:self.final_k]
        
        # Get chunks
        final_chunks = [self.chunk_texts[idx] for idx in mmr_indices if idx < len(self.chunk_texts)]
        
        if not final_chunks:
            return {"error": "No chunks retrieved."}
        
        # Expand context with additional relevant chunks
        expanded_context = self._expand_context(question, final_chunks, max_chunks=5)
        
        # Generate answer with LLM - improved prompt
        answer = None
        if self.llm_available:
            try:
                prompt = f"""Context:
{expanded_context}

Question: {question}

Please provide a clear and accurate answer based only on the context above. If the context does not contain the answer, say "I cannot find the answer in the document."

Answer:"""
                
                inputs = self.tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
                if self.device == "cuda":
                    inputs = {k: v.to(self.device) for k, v in inputs.items()}
                
                outputs = self.model.generate(
                    **inputs, 
                    max_new_tokens=150,  # Increased for better answers
                    temperature=0.1,
                    do_sample=True
                )
                answer = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                # Extract answer part
                if "Answer:" in answer:
                    answer = answer.split("Answer:")[-1].strip()
                else:
                    answer = answer.strip()
                
            except Exception as e:
                print(f"LLM generation failed: {e}")
                answer = None
        
        if answer is None or not answer:
            answer = final_chunks[0] if final_chunks else "No answer found."
        
        elapsed_time = time.time() - start_time
        
        # Build citations
        citations = []
        for i, idx in enumerate(mmr_indices[:self.final_k]):
            if idx < len(self.chunk_texts):
                chunk_text = self.chunk_texts[idx]
                citations.append({
                    "chunk_id": i + 1,
                    "text": chunk_text[:200] + "..." if len(chunk_text) > 200 else chunk_text,
                    "length": len(chunk_text)
                })
        
        return {
            "question": question,
            "answer": answer,
            "citations": citations,
            "num_citations": len(citations),
            "elapsed_time": elapsed_time,
            "num_chunks_retrieved": len(mmr_indices)
        }

# Re-initialize the RAG system with accuracy improvements
print("\nRe-initializing RAG System with accuracy improvements...")
rag_system = RAGSystem()
print("RAG System ready!")


Re-initializing RAG System with accuracy improvements...
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded. Dimension: 384
Loading LLM...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


LLM loaded successfully: google/flan-t5-small
RAG System ready!


In [16]:
def process_uploaded_pdf(file):
    """
    Process uploaded PDF file.
    
    Args:
        file: Uploaded file object from Gradio
    
    Returns:
        String status message
    """
    if file is None:
        return "Please upload a PDF file."
    
    try:
        # Handle file from Gradio
        if hasattr(file, 'name'):
            file_path = file.name
        else:
            # Save uploaded file
            with tempfile.NamedTemporaryFile(delete=False, suffix='.pdf') as tmp_file:
                tmp_file.write(file)
                file_path = tmp_file.name
        
        # Process PDF
        result = rag_system.process_pdf(file_path)
        
        # Clean up temp file
        if not hasattr(file, 'name'):
            os.unlink(file_path)
        
        if result["success"]:
            return f"""PDF processed successfully!

Chunks created: {result['num_chunks']}
Embedding dimension: {result['embedding_dimension']}
Text length: {result['text_length']} characters
Processing time: {result['elapsed_time']:.2f} seconds

You can now ask questions about this document."""
        else:
            return f"Error: {result['error']}"
    
    except Exception as e:
        return f"Error processing file: {str(e)}"

In [17]:
def answer_question(question, history):
    """
    Answer a question using the RAG system.
    
    Args:
        question: User question
        history: Chat history (list of tuples)
    
    Returns:
        Tuple of (question, answer, chat history)
    """
    if not question or not question.strip():
        return "", "", history
    
    if history is None:
        history = []
    
    # Get RAG response
    result = rag_system.query(question)
    
    if "error" in result:
        return "", f"Error: {result['error']}", history
    
    # Format answer
    answer = result['answer']
    
    # Add citations
    if result['citations']:
        answer += "\n\nSources:"
        for citation in result['citations']:
            answer += f"\n\n{citation['chunk_id']}. {citation['text']}"
    
    answer += f"\n\nRetrieval time: {result['elapsed_time']:.2f} seconds"
    
    history.append((question, answer))
    
    return "", "", history

In [18]:
print("Creating Gradio Interface...")

with gr.Blocks(
    title="RAG Document QA System",
    theme=gr.themes.Soft()
) as demo:
    
    gr.Markdown("""
    # RAG Document QA System
    
    Upload a PDF document and ask questions about its content.
    
    The system uses:
    - Hybrid Retrieval: FAISS (Dense) + BM25 (Sparse) + RRF Fusion
    - Reranking: MMR for diverse results
    - Generation: Flan-T5-Small for answers
    - Citations: Source tracking for transparency
    
    ---
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Upload Document")
            pdf_upload = gr.File(
                label="Upload PDF",
                file_types=[".pdf"],
                file_count="single"
            )
            
            upload_button = gr.Button("Process PDF", variant="primary")
            status_output = gr.Textbox(
                label="Status",
                lines=5,
                interactive=False,
                value="Upload a PDF to get started."
            )
            
            gr.Markdown("""
            ---
            ### Instructions
            1. Upload a PDF document
            2. Click Process PDF
            3. Wait for processing confirmation
            4. Ask questions in the chat
            """)
        
        with gr.Column(scale=2):
            gr.Markdown("### Ask Questions")
            
            chatbot = gr.Chatbot(
                label="Q&A Chat",
                height=400
            )
            
            question_input = gr.Textbox(
                label="Your Question",
                placeholder="Ask a question about the document...",
                lines=2
            )
            
            with gr.Row():
                ask_button = gr.Button("Ask Question", variant="primary")
                clear_button = gr.Button("Clear Chat")
    
    # Event handlers
    upload_button.click(
        fn=process_uploaded_pdf,
        inputs=pdf_upload,
        outputs=status_output
    )
    
    ask_button.click(
        fn=answer_question,
        inputs=[question_input, chatbot],
        outputs=[question_input, chatbot, chatbot]
    )
    
    question_input.submit(
        fn=answer_question,
        inputs=[question_input, chatbot],
        outputs=[question_input, chatbot, chatbot]
    )
    
    clear_button.click(
        fn=lambda: [],
        inputs=None,
        outputs=chatbot,
        queue=False
    )
    
    clear_button.click(
        fn=lambda: "",
        inputs=None,
        outputs=question_input,
        queue=False
    )

print("Gradio interface created successfully!")

Creating Gradio Interface...
Gradio interface created successfully!


In [19]:
import socket

def find_free_port():
    """Find a free port to use."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        return s.getsockname()[1]

print("=" * 70)
print("LAUNCHING GRADIO INTERFACE")
print("=" * 70)

# Find a free port
port = find_free_port()
print(f"\nUsing port: {port}")

print("\nLaunching Gradio interface...")
print(f"Local URL: http://localhost:{port}")
print("=" * 70)

try:
    demo.launch(
        server_name="0.0.0.0",
        server_port=port,
        share=True,
        debug=False
    )
except Exception as e:
    print(f"Error with share=True: {e}")
    print("Trying local only...")
    demo.launch(
        server_name="0.0.0.0",
        server_port=port,
        share=False,
        debug=True
    )

LAUNCHING GRADIO INTERFACE

Using port: 43999

Launching Gradio interface...
Local URL: http://localhost:43999
* Running on local URL:  http://0.0.0.0:43999
* Running on public URL: https://c578642da346bbd65f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error


Processing PDF: /tmp/gradio/b147fb3b129f883630a7b17bbebdd9b27a96618a0dd8452be8fb29c9d1ec341e/Internship_Plan_Basic.pdf
Created 14 chunks
Total text length: 3483 characters
Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Building FAISS index...
Building BM25 index...

Processing PDF: /tmp/gradio/b147fb3b129f883630a7b17bbebdd9b27a96618a0dd8452be8fb29c9d1ec341e/Internship_Plan_Basic.pdf
Created 14 chunks
Total text length: 3483 characters
Generating embeddings...


...

Building FAISS index...
Building BM25 index...


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error


Processing PDF: /tmp/gradio/387c98741fc4189895ebdeb407c8492c7e3717b6e717b580693a76e415cf46a5/Sanaullah.pdf
Created 8 chunks
Total text length: 1946 characters
Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Building FAISS index...
Building BM25 index...
